# Quantum Playground — Exploración interactiva

Notebook complementario al código de `src/`. Aquí puedes tocar parámetros y reejecutar celdas.

**Requisitos:** ver `requirements.txt`.

## Contenido
1. Estados de Bell
2. GHZ
3. Teleportación cuántica
4. Grover
5. Deutsch–Jozsa
6. QFT
7. VQE para H₂

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector, plot_state_qsphere

sim = AerSimulator()
print('Qiskit cargado.')

## 1) Bell — |Φ+⟩

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0); qc.cx(0, 1)
sv = Statevector.from_instruction(qc)
display(qc.draw('mpl'))
display(plot_state_qsphere(sv))

qc.measure([0, 1], [0, 1])
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
plot_histogram(counts)

## 2) GHZ — tweakea `n`

In [ ]:
from ghz_state import ghz
n = 4
qc = ghz(n)
qc.measure(range(n), range(n))
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
plot_histogram(counts, title=f'GHZ {n} qubits')

## 3) Teleportación

In [ ]:
from teleportation import teleportation_circuit
qc = teleportation_circuit()
qc.draw('mpl', fold=-1)

In [ ]:
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
plot_histogram(counts)

## 4) Grover — cambia `marked`

In [ ]:
from grover import grover
n = 4
marked = '1101'  # cambia aquí
qc, iters = grover(n, marked)
print(f'iteraciones: {iters}')
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
plot_histogram(counts, title=f'Grover -> |{marked}>')

## 5) Deutsch–Jozsa

In [ ]:
from deutsch_jozsa import dj_circuit, constant_oracle, balanced_oracle
n = 4
qc_c = dj_circuit(n, constant_oracle(n, value=0))
qc_b = dj_circuit(n, balanced_oracle(n, '1100'))
c1 = sim.run(transpile(qc_c, sim), shots=2048).result().get_counts()
c2 = sim.run(transpile(qc_b, sim), shots=2048).result().get_counts()
print('constante :', c1)
print('balanceada:', c2)

## 6) QFT

In [ ]:
from qft import qft
n = 3
prep = QuantumCircuit(n)
prep.x(0)  # |x=1>
full = prep.compose(qft(n))
sv = Statevector.from_instruction(full)
display(full.draw('mpl'))
plot_state_qsphere(sv)

## 7) VQE H₂

In [ ]:
from vqe_h2 import expectation, h2_hamiltonian
from scipy.optimize import minimize

H = h2_hamiltonian()
history = []
def cost(p):
    e = expectation(p, H); history.append(e); return e

np.random.seed(42)
x0 = np.random.uniform(0, 2*np.pi, 4)
res = minimize(cost, x0, method='COBYLA', options={'maxiter': 200})
print(f'E_min = {res.fun:.6f} Ha   (exacta: -1.85728 Ha)')

plt.plot(history); plt.axhline(-1.85728, ls='--', color='red')
plt.xlabel('iter'); plt.ylabel('E (Ha)'); plt.grid(alpha=.3); plt.title('VQE H2')